# Analysis – Behavior
**Michon Linde et al., Nature Communications**  
*"The Intermediate Hippocampus Integrates Shock-Observation and Spatial Information during Observational Fear Memory"*

Covers: **Fig. 1b–c** · **Extended Data Fig. 1a–g**

> Set `Folder_path` below to the directory containing the summary data tables.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.api as sm
from statsmodels.formula.api import ols

import seaborn as sns
from matplotlib import pyplot as plt

from Toolbox.statistics.core import bootstrap

## Helper functions

In [ ]:
def cutoff_1d(x, fac=0.5):
    """
    Identify the local minimum of a kernel density estimate of x.
    Used to threshold the differential contextual freezing score (shock - safe)
    and separate RECALLER from NON-RECALLER animals (Fig. 1c).

    Parameters
    ----------
    x   : 1-D array of values (here: contrast_immo per animal)
    fac : KDE bandwidth factor (Scott's Rule multiplier). Default 0.5.

    Returns
    -------
    threshold : float — the first local minimum of the KDE.
    """
    from scipy import stats
    from functools import partial

    def my_kde_bandwidth(obj, fac=1.0 / 5):
        """Scott's Rule multiplied by a constant factor."""
        return np.power(obj.n, -1.0 / (obj.d + 4)) * fac

    kde = stats.gaussian_kde(x, bw_method=partial(my_kde_bandwidth, fac=fac))
    x_eval = np.linspace(x.min() - 1, x.max() + 1, 500)

    import Toolbox.signals.core as sig
    threshold = sig.localminima(kde(x_eval), x=x_eval, method='gradient')[0][0]
    return threshold

## Data loading

In [ ]:
# ── Set this path to the folder containing the summary data tables ──────────
Folder_path = "/data07/Fred/Ctx_Hpc/summaries/NatCom/"

# Freezing behavior: one row per rat × context × phase
df = pd.read_csv(os.path.join(Folder_path, 'table_behavior_freezing.csv'))

# Spatial occupancy and demonstrator-vicinity per rat × session
df_occup = pd.read_csv(os.path.join(Folder_path, 'table_behavior_vicinity.csv'))
df_occup.proximity = [np.fromstring(p.strip('[]'), sep=' ') for p in df_occup.proximity]

# Peri-shock behavioral time series (pupil, eye, speed, distance to divider)
df_ShockObs = pd.read_parquet(os.path.join(Folder_path, 'table_behavior_shockobs.parquet'))

print(f"Loaded {df['rat'].nunique()} observer animals.")
print(f"  df           : {df.shape[0]} rows — freezing behavior")
print(f"  df_occup     : {df_occup.shape[0]} rows — demonstrator vicinity & occupancy")
print(f"  df_ShockObs  : {df_ShockObs.shape[0]} rows — peri-shock time series")

---
## Figure 1b
**Differential contextual freezing at recall.**  
Percentage change in immobility (recall minus solo baseline) in the safe vs. shock context, across all 14 observer animals.  
*Two-sided paired t-test; Bayes factor BF₁₀ reported.*

In [ ]:
var = 'immobile_idx'

fig, ax = plt.subplots(1, 1, figsize=(1.5, 4))

# Paired lines across animals
lines = (df.query("phase == 'test'")
           .set_index(['rat', 'phase', 'condition'])
           .unstack()
           .reset_index()[var])
for l in np.array(lines):
    ax.plot([0, 1], l, color='grey', alpha=0.5)

ax.axhline(0, color='grey', ls='--')

sns.boxplot(x='condition', y=var, data=df.query("phase == 'test'"),
            order=['ctrl', 'shock'],
            palette=['darkgreen', 'rebeccapurple'],
            boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax)
sns.stripplot(x='condition', y=var, data=df.query("phase == 'test'"),
              order=['ctrl', 'shock'],
              palette=['darkgreen', 'rebeccapurple'],
              size=10, alpha=0.8, ax=ax)

ax.set(ylim=(-7.5, 20),
       xticklabels=['safe', 'shock'],
       xlabel='',
       ylabel='immobility change (% time)')
sns.despine(offset=True, trim=True)

### Statistics — Fig. 1b | Paired t-test: shock vs. safe context (recall)

In [ ]:
# Paired two-sided t-test: change in immobility, shock vs. safe context at recall
var = 'immobile_idx'
results = pg.ttest(
    df.query("phase == 'test' and condition == 'shock'")[var],
    df.query("phase == 'test' and condition == 'ctrl'")[var],
    paired=True)

print("Fig. 1b — Paired t-test: immobility change, shock vs. safe context at recall")
print(results.to_string())

---
## Figure 1c
**Inter-individual variability in contextual freezing recall.**  
Top: distribution of the differential freezing score Δ(shock − safe) with the KDE-derived threshold  
separating NON-RECALLER (grey) from RECALLER (orchid) animals.  
Bottom: same data as Fig. 1b, split by recall status.

In [ ]:
var = 'immobile_idx'

# ── Distribution of Δ(shock − safe) with RECALLER/NON-RECALLER threshold ─────
fig, ax = plt.subplots(1, 1, figsize=(4, 0.5))

ax.axvline(0, color='grey', ls='--')
ax.axvline(cutoff_1d(df.query("session == 'test_shock'")['contrast_immo'], fac=0.5),
           color='darkorange', ls='--', alpha=0.5, label='threshold')

sns.stripplot(x='contrast_immo',
              data=df.query("session == 'test_shock' and context_learning == 'not learned'"),
              y='session', palette=['grey'], size=10, alpha=0.8, ax=ax)
sns.stripplot(x='contrast_immo',
              data=df.query("session == 'test_shock' and context_learning == 'learned'"),
              y='session', palette=['orchid'], size=10, alpha=0.8, ax=ax)

ax.spines['bottom'].set_visible(False)
ax.set(xlim=(-20, 20), yticks=[],
       xlabel='', ylabel='',
       title=u'Δ(shock − safe) immobility')
sns.despine(offset=True, trim=True, left=True, right=True, bottom=True, top=False)

# ── Freezing change split by recall status (NON-RECALLER | RECALLER) ──────────
fig, ax = plt.subplots(1, 2, figsize=(3, 3), sharey=True, sharex=True)

for l, (learn, label) in enumerate([('not learned', 'NON-RECALLER'), ('learned', 'RECALLER')]):
    lines = (df.query("phase == 'test' and context_learning == '{}'".format(learn))
               .set_index(['rat', 'phase', 'condition'])
               .unstack()
               .reset_index()[var])
    for c in np.array(lines):
        ax[l].plot([0, 1], c, color='grey', alpha=0.5)

    ax[l].axhline(0, color='grey', ls='--')
    sns.boxplot(x='condition', y=var,
                data=df.query("phase == 'test' and context_learning == '{}'".format(learn)),
                order=['ctrl', 'shock'],
                palette=['darkgreen', 'rebeccapurple'],
                boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax[l])
    sns.stripplot(x='condition', y=var,
                  data=df.query("phase == 'test' and context_learning == '{}'".format(learn)),
                  order=['ctrl', 'shock'],
                  palette=['darkgreen', 'rebeccapurple'],
                  size=10, alpha=0.8, ax=ax[l])
    ax[l].set(ylim=(-7.5, 20),
              xticklabels=['safe', 'shock'],
              xlabel='', ylabel='', title=label)

ax[0].set(ylabel='immobility change (% time)')
sns.despine(offset=True, trim=True)

---
## Extended Data Figure 1a
**No difference in freezing during shock observation between RECALLER and NON-RECALLER animals.**  
Percentage of time spent immobile during the shock-observation phase.

In [ ]:
var = '%time immobile'

fig, ax = plt.subplots(1, 1, figsize=(1.0, 4))

sns.boxplot(y=var, data=df.query("session == 'shock_observation'"),
            x='context_learning', order=['not learned', 'learned'],
            palette=['grey', 'orchid'], fliersize=0.0,
            boxprops=dict(alpha=0.6))
sns.stripplot(y=var, data=df.query("session == 'shock_observation'"),
              x='context_learning', order=['not learned', 'learned'],
              palette=['grey', 'orchid'], dodge=False, size=10, alpha=0.6)

ax.set(ylim=(0, 55),
       xticklabels=['NON-RECALLER', 'RECALLER'],
       xlabel='', ylabel='immobility (% time)')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 1a | Freezing during shock observation

In [ ]:
var = '%time immobile'

# One-sample t-test: immobility > 0 during shock observation (all animals)
results_vs0 = pg.ttest(df.query("session == 'shock_observation'")[var], 0)
print("Ext. Data Fig. 1a — One-sample t-test: immobility > 0 during shock observation")
print(results_vs0.to_string())

# Independent t-test: RECALLER vs. NON-RECALLER
results_grp = pg.ttest(
    df.query("session == 'shock_observation' and context_learning == 'learned'")[var],
    df.query("session == 'shock_observation' and context_learning == 'not learned'")[var],
    paired=False)
print("\nExt. Data Fig. 1a — Independent t-test: RECALLER vs. NON-RECALLER")
print(results_grp.to_string())

---
## Extended Data Figure 1b
**Observers spend the majority of time near the demonstrator's divider during shock observation.**  
Mean occupancy as a function of distance from the divider, separately for RECALLER (orchid) and  
NON-RECALLER (grey) animals. Shaded areas: 95% CI (500 bootstrap iterations).

In [ ]:
phase = 'shock_observation'

prox_rec    = np.vstack(df_occup.query("session == '{}' and context_learning == 'learned'".format(phase))['proximity'])
prox_notrec = np.vstack(df_occup.query("session == '{}' and context_learning == 'not learned'".format(phase))['proximity'])

ci_rec    = bootstrap.ci(prox_rec,    axis=0, statistic=lambda x: np.nanmean(x, axis=0))
ci_notrec = bootstrap.ci(prox_notrec, axis=0, statistic=lambda x: np.nanmean(x, axis=0))

proxi = (np.arange(20) + 0.5) * 2.5   # bin centres in cm

fig, ax = plt.subplots(1, 1, figsize=(1.5, 4))

ax.plot(np.nanmean(prox_notrec, axis=0), proxi,
        color='grey', label='NON-RECALLER', lw=2.0, alpha=0.8)
ax.fill_betweenx(proxi, ci_notrec[0], ci_notrec[1], color='grey', alpha=0.4)

ax.plot(np.nanmean(prox_rec, axis=0), proxi,
        color='orchid', label='RECALLER', lw=2.0, alpha=0.8)
ax.fill_betweenx(proxi, ci_rec[0], ci_rec[1], color='orchid', alpha=0.4)

ax.legend(frameon=False)
ax.set(xlim=(0, 40), ylim=(-0.5, 48), yticks=[0, 50],
       xlabel='occupancy (% time)', ylabel='distance to divider (cm)')
ax.invert_yaxis()
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 1b | ANCOVA: occupancy × distance × recall status

In [ ]:
# ANCOVA: effect of distance-to-divider on occupancy, controlling for recall status
df_prox = pd.DataFrame({
    'prox': np.concatenate([np.ravel(prox_rec), np.ravel(prox_notrec)]),
    'distance': np.concatenate([
        np.vstack(np.arange(prox_rec.shape[1]).tolist() * prox_rec.shape[0]).ravel(),
        np.vstack(np.arange(prox_notrec.shape[1]).tolist() * prox_notrec.shape[0]).ravel()
    ]),
    'recall_status': np.concatenate([
        np.array(['RECALLER']     * np.ravel(prox_rec).shape[0]),
        np.array(['NON-RECALLER'] * np.ravel(prox_notrec).shape[0])
    ])
})

model  = ols('prox ~ distance + C(recall_status) + distance:C(recall_status)',
             data=df_prox).fit()
result = sm.stats.anova_lm(model, type=2)

print("Ext. Data Fig. 1b — ANCOVA: occupancy ~ distance × recall status")
print(result.to_string())
print()
print(model.params)

---
## Extended Data Figure 1c
**Stronger pupil dilation and eye-movement responses to footshock delivery in RECALLER animals.**  
Left: mean z-scored pupil diameter and eye movement around footshock delivery (shaded area: 1 s shock window).  
Right: boxplots of the mean response difference (3 s post − 3 s pre shock onset).

In [ ]:
# ── Time courses: pupil diameter and eye movement ─────────────────────────────
t = np.vstack(df_ShockObs.query("context_learning == 'learned'")['time_HM'].dropna())[0]

for var in ['pupil diameter (z)', 'eye movement (z)']:
    beh_rec    = np.vstack(df_ShockObs.query("context_learning == 'learned'")[var].dropna())
    beh_notrec = np.vstack(df_ShockObs.query("context_learning == 'not learned'")[var].dropna())

    ci_rec    = bootstrap.ci(beh_rec,    nsamples=500, alpha=0.05,
                             statistic=lambda x: np.nanmean(x, axis=0))
    ci_notrec = bootstrap.ci(beh_notrec, nsamples=500, alpha=0.05,
                             statistic=lambda x: np.nanmean(x, axis=0))

    fig = plt.figure(figsize=(2, 4))
    plt.axvspan(0, 1, color='gold', alpha=0.25, label='shock (1 s)')
    plt.fill_between(t, ci_notrec[0], ci_notrec[1], alpha=0.2, color='grey')
    plt.plot(t, np.nanmean(beh_notrec, axis=0), lw=1.5, color='grey',   label='NON-RECALLER')
    plt.fill_between(t, ci_rec[0],    ci_rec[1],    alpha=0.2, color='orchid')
    plt.plot(t, np.nanmean(beh_rec,    axis=0), lw=1.5, color='orchid', label='RECALLER')
    plt.legend(frameon=False, loc='upper left')
    plt.gca().set(ylim=(-1.5, 5), xlim=(-3, 5),
                  xlabel='time from shock onset (s)', ylabel=var)

# ── Boxplots of mean post-shock response ─────────────────────────────────────
metric = 'rep'
for var in ['pupil_diameter', 'eye_movement']:
    label = 'pupil diameter (z)' if var == 'pupil_diameter' else 'eye movement (z)'
    fig, ax = plt.subplots(1, 1, figsize=(1, 3))
    sns.boxplot(x='context_learning', y='{}_{}'.format(metric, var),
                data=df_ShockObs,
                order=['not learned', 'learned'],
                palette=['grey', 'orchid'],
                boxprops=dict(alpha=0.3), fliersize=0.0, ax=ax)
    sns.stripplot(x='context_learning', y='{}_{}'.format(metric, var),
                  data=df_ShockObs,
                  order=['not learned', 'learned'],
                  palette=['grey', 'orchid'],
                  size=10, alpha=0.8, ax=ax)
    ax.set(ylim=(-1.5, 4),
           xticklabels=['NON-RECALLER', 'RECALLER'],
           xlabel='', ylabel=label)

### Statistics — Extended Data Fig. 1c | Independent t-tests: pupil & eye response (RECALLER vs. NON-RECALLER)

In [ ]:
metric = 'rep'

for var, label in [('pupil_diameter', 'Pupil diameter'), ('eye_movement', 'Eye movement')]:
    ttest = pg.ttest(
        df_ShockObs.query("context_learning == 'learned'")['{}_{}'.format(metric, var)].dropna(),
        df_ShockObs.query("context_learning == 'not learned'")['{}_{}'.format(metric, var)].dropna())
    print("Ext. Data Fig. 1c — Independent t-test: {} (RECALLER vs. NON-RECALLER)".format(label))
    print(ttest.to_string())
    print()

---
## Extended Data Figure 1d
**Similar orienting speed response to footshock delivery in RECALLER and NON-RECALLER animals.**  
Mean head speed around footshock delivery. Shaded area: 1 s shock window.

In [ ]:
t = np.vstack(df_ShockObs.query("context_learning == 'learned'")['time'])[0]

# ── Speed time course ─────────────────────────────────────────────────────────
beh_rec    = np.vstack(df_ShockObs.query("context_learning == 'learned'")['speed'])
beh_notrec = np.vstack(df_ShockObs.query("context_learning == 'not learned'")['speed'])

ci_rec    = bootstrap.ci(beh_rec,    nsamples=500, alpha=0.05,
                         statistic=lambda x: np.nanmean(x, axis=0))
ci_notrec = bootstrap.ci(beh_notrec, nsamples=500, alpha=0.05,
                         statistic=lambda x: np.nanmean(x, axis=0))

fig = plt.figure(figsize=(2, 4))
plt.axvspan(0, 1, color='gold', alpha=0.25, label='shock (1 s)')
plt.fill_between(t, ci_notrec[0], ci_notrec[1], alpha=0.2, color='grey')
plt.plot(t, np.nanmean(beh_notrec, axis=0), lw=1.5, color='grey',   label='NON-RECALLER')
plt.fill_between(t, ci_rec[0],    ci_rec[1],    alpha=0.2, color='orchid')
plt.plot(t, np.nanmean(beh_rec,    axis=0), lw=1.5, color='orchid', label='RECALLER')
plt.legend(frameon=False, loc='upper left')
plt.gca().set(ylim=(0, 25), xlim=(-3, 5),
              xlabel='time from shock onset (s)', ylabel='head speed (cm/s)')

# ── Boxplot of mean post-shock speed ─────────────────────────────────────────
metric = 'rep'
fig, ax = plt.subplots(1, 1, figsize=(1, 3))
sns.boxplot(x='context_learning', y='{}_speed'.format(metric),
            data=df_ShockObs, order=['not learned', 'learned'],
            palette=['grey', 'orchid'],
            boxprops=dict(alpha=0.3), fliersize=0.0, ax=ax)
sns.stripplot(x='context_learning', y='{}_speed'.format(metric),
              data=df_ShockObs, order=['not learned', 'learned'],
              palette=['grey', 'orchid'],
              size=10, alpha=0.8, ax=ax)
ax.set(ylim=(0, 7.5),
       xticklabels=['NON-RECALLER', 'RECALLER'],
       xlabel='', ylabel='mean speed post-shock (cm/s)')

### Statistics — Extended Data Fig. 1d | Independent t-test: orienting speed (RECALLER vs. NON-RECALLER)

In [ ]:
metric = 'rep'
ttest = pg.ttest(
    df_ShockObs.query("context_learning == 'learned'")['rep_speed'].dropna(),
    df_ShockObs.query("context_learning == 'not learned'")['rep_speed'].dropna())
print("Ext. Data Fig. 1d — Independent t-test: orienting speed (RECALLER vs. NON-RECALLER)")
print(ttest.to_string())

---
## Extended Data Figure 1e
**Similar distance-to-divider response to footshock delivery in RECALLER and NON-RECALLER animals.**  
Mean distance of the observer's head to the demonstrator's divider around footshock delivery.

In [ ]:
t = np.vstack(df_ShockObs.query("context_learning == 'learned'")['time'])[0]

# ── Distance time course ──────────────────────────────────────────────────────
beh_rec    = np.vstack(df_ShockObs.query("context_learning == 'learned'")['distance'])
beh_notrec = np.vstack(df_ShockObs.query("context_learning == 'not learned'")['distance'])

ci_rec    = bootstrap.ci(beh_rec,    nsamples=500, alpha=0.05,
                         statistic=lambda x: np.nanmean(x, axis=0))
ci_notrec = bootstrap.ci(beh_notrec, nsamples=500, alpha=0.05,
                         statistic=lambda x: np.nanmean(x, axis=0))

fig = plt.figure(figsize=(2, 4))
plt.axvspan(0, 1, color='gold', alpha=0.25, label='shock (1 s)')
plt.fill_between(t, ci_notrec[0], ci_notrec[1], alpha=0.2, color='grey')
plt.plot(t, np.nanmean(beh_notrec, axis=0), lw=1.5, color='grey',   label='NON-RECALLER')
plt.fill_between(t, ci_rec[0],    ci_rec[1],    alpha=0.2, color='orchid')
plt.plot(t, np.nanmean(beh_rec,    axis=0), lw=1.5, color='orchid', label='RECALLER')
plt.legend(frameon=False, loc='upper left')
plt.gca().set(ylim=(0, 15), xlim=(-3, 5),
              xlabel='time from shock onset (s)', ylabel='distance to divider (cm)')

# ── Boxplot of mean post-shock distance change ────────────────────────────────
metric = 'rep'
fig, ax = plt.subplots(1, 1, figsize=(1, 3))
sns.boxplot(x='context_learning', y='{}_distance'.format(metric),
            data=df_ShockObs, order=['not learned', 'learned'],
            palette=['grey', 'orchid'],
            boxprops=dict(alpha=0.3), fliersize=0.0, ax=ax)
sns.stripplot(x='context_learning', y='{}_distance'.format(metric),
              data=df_ShockObs, order=['not learned', 'learned'],
              palette=['grey', 'orchid'],
              size=10, alpha=0.8, ax=ax)
ax.set(ylim=(-6, 6),
       xticklabels=['NON-RECALLER', 'RECALLER'],
       xlabel='', ylabel='mean distance change (cm)')

### Statistics — Extended Data Fig. 1e | Independent t-test: distance-to-divider (RECALLER vs. NON-RECALLER)

In [ ]:
metric = 'rep'
ttest = pg.ttest(
    df_ShockObs.query("context_learning == 'learned'")['rep_distance'].dropna(),
    df_ShockObs.query("context_learning == 'not learned'")['rep_distance'].dropna())
print("Ext. Data Fig. 1e — Independent t-test: distance-to-divider (RECALLER vs. NON-RECALLER)")
print(ttest.to_string())

---
## Extended Data Figure 1f
**Higher propensity to approach the demonstrator's compartment in RECALLER animals — baseline with demonstrator.**  
Percentage of time spent within 7.5 cm of the demonstrator's divider during the duo baseline phase  
(both safe and shock contexts), split by recall status.

In [ ]:
var = 'demo_vicinity'

fig, ax = plt.subplots(1, 1, figsize=(2, 4))

sns.boxplot(y=var, data=df_occup.query("phase == 'context_demo'"),
            x='condition', hue='context_learning',
            hue_order=['not learned', 'learned'],
            palette=['grey', 'orchid'], fliersize=0.0,
            boxprops=dict(alpha=0.6))
sns.stripplot(y=var, data=df_occup.query("phase == 'context_demo'"),
              x='condition', hue='context_learning',
              hue_order=['not learned', 'learned'],
              palette=['grey', 'orchid'],
              dodge=True, size=10, alpha=0.6)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['NON-RECALLER', 'RECALLER'], frameon=False)
ax.set(ylim=(0, 80),
       xticklabels=['safe', 'shock'],
       xlabel='', ylabel='demonstrator vicinity (% time)')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 1f | ANOVA: demonstrator vicinity during duo baseline

In [ ]:
var     = 'demo_vicinity'
factor1 = 'condition'
factor2 = 'context_learning'
data    = df_occup.query("phase == 'context_demo'")[[var, factor1, factor2]]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Ext. Data Fig. 1f — Two-way ANOVA: demonstrator vicinity (duo baseline) ~ context × recall status")
print()
print(results.to_string())
print()
print(posthoc1.to_string())

---
## Extended Data Figure 1g
**Higher propensity to approach the demonstrator's compartment in RECALLER animals — recall phase.**  
Percentage of time spent within 7.5 cm of the demonstrator's divider during the recall phase  
(both safe and shock contexts), split by recall status.

In [ ]:
var = 'demo_vicinity'

fig, ax = plt.subplots(1, 1, figsize=(2, 4))

sns.boxplot(y=var, data=df_occup.query("phase == 'test'"),
            x='condition', hue='context_learning',
            hue_order=['not learned', 'learned'],
            palette=['grey', 'orchid'], fliersize=0.0,
            boxprops=dict(alpha=0.6))
sns.stripplot(y=var, data=df_occup.query("phase == 'test'"),
              x='condition', hue='context_learning',
              hue_order=['not learned', 'learned'],
              palette=['grey', 'orchid'],
              dodge=True, size=10, alpha=0.6)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['NON-RECALLER', 'RECALLER'], frameon=False)
ax.set(ylim=(0, 80),
       xticklabels=['safe', 'shock'],
       xlabel='', ylabel='demonstrator vicinity (% time)')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 1g | ANOVA: demonstrator vicinity during recall

In [ ]:
var     = 'demo_vicinity'
factor1 = 'condition'
factor2 = 'context_learning'
data    = df_occup.query("phase == 'test'")[[var, factor1, factor2]]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Ext. Data Fig. 1g — Two-way ANOVA: demonstrator vicinity (recall) ~ context × recall status")
print()
print(results.to_string())
print()
print(posthoc1.to_string())